In [1]:
from rag_helper import RAGBase
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from ingest import load_faq_data
from rag_helper import RAGBase
import os

In [3]:
from groq import Groq
groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
import time
from sqlitesearch import TextSearchIndex

index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

# INDEXING - RUN ONCE ONLY (executed)

In [5]:
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")

Loaded 1350 documents


Filter to just the LLM Zoomcamp documents:

In [6]:
docs_llm = [doc for doc in documents if doc["course"] == "llm-zoomcamp"]
print(f"LLM Zoomcamp: {len(docs_llm)} documents")

LLM Zoomcamp: 85 documents


In [ ]:
for doc in docs_llm:
    index.add(doc)
    print(f"""Added: {doc["question"][:60]}...""")
    time.sleep(0.5)

index.close()
print("Done. Index saved to faq.db")

# RAG ASSISTANT

In [8]:
assistant = RAGBase(
    index=index,
    llm_client=groq_client,
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

The answer to your question is: **Yes**, you can still join the course. If you're interested in receiving a certificate, make sure to submit your project while submissions are still being accepted. You don't need a confirmation email to start learning and submitting homework. Just start with the course materials and you can submit your work as you go along.


In [5]:
custom_instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=groq_client,
    instructions=custom_instructions,
)

In [6]:
assistant.rag("How do I get a certificate?")

'It seems like you\'re interested in obtaining a certificate for a course. Based on the provided context, here are the key points to consider:\n\n1. **Missing Homework**: You can still get a certificate even if you missed the first homework. The homework is not mandatory but is recommended for reinforcing concepts.\n\n2. **Self-Paced Mode**: You cannot get a certificate by following the course in self-paced mode. You need to finish the course with a "live" cohort.\n\n3. **Peer Review**: To get a certificate, you need to peer-review 3 capstones after submitting your project. This can only be done during the course run.\n\n4. **Course Enrollment Timing**: You can still join the course, but for certificate eligibility, ensure you submit your project while submissions are being accepted.\n\nTo directly answer your question: \n- Ensure you are enrolled in a "live" cohort.\n- Complete and submit your Capstone project.\n- Peer-review 3 capstone projects after submitting your own.\n\nIf you fo

In [7]:
assistant.rag("Can I still join the course after it started?")

'Based on the provided context, the answer to your question is:\n\n**Yes**, you can still join the course after it started. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. Additionally, keep in mind that you will need to participate in the "live" cohort to be eligible for a certificate, as self-paced mode does not award certificates.'

In [8]:
assistant.rag("How do I run Ollama locally?")

'## Running Ollama Locally\n\nTo run Ollama locally, follow these steps:\n\n### Step 1: Install Ollama\n\nFirst, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n* **macOS**: Download the `.pkg` and install it.\n* **Windows**: Download the `.msi` and install it.\n* **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\n### Step 2: Run Ollama\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n* Download the LLaMA 3 model (~4GB).\n* Start the model locally.\n* Open a chat-like interface where you can type questions.\n\n### Step 3: Test the Ollama Local Server\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\n### Step 4: Install the Python Client\n\nT